# Does Weather Affect the Stock Market? A BIST 100 Analysis
**DSA210 – Data Collection & EDA**

## 1. BIST 100 Data

In [1]:
import yfinance as yf
import pandas as pd
import os

os.makedirs("../data", exist_ok=True)

bist = yf.download("XU100.IS", start="2023-01-01", end="2024-12-31")
bist = bist[["Open", "Close", "Volume"]].copy()
bist.columns = ["open", "close", "volume"]
bist["daily_return"] = (bist["close"] - bist["open"]) / bist["open"] * 100
bist.index = pd.to_datetime(bist.index)
bist.index.name = "date"

bist.to_csv("../data/bist100.csv")
print("Saved:", len(bist), "trading days")
bist.head()

[*********************100%***********************]  1 of 1 completed

Saved: 496 trading days


,open,close,volume,daily_return
date,,,,
2023-01-02,5568.370125,5661.069824,4834734400,1.664755
2023-01-03,5697.569937,5626.570312,5459192700,-1.246139
2023-01-04,5646.570148,5523.470703,4992602800,-2.180075
2023-01-05,5555.970307,5116.372559,4510533000,-7.912169
2023-01-06,5120.972851,5341.971680,5331359700,4.315563


## 2. Istanbul Weather Data

In [2]:
import requests

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 41.0082,
    "longitude": 28.9784,
    "start_date": "2023-01-01",
    "end_date": "2024-12-31",
    "daily": "temperature_2m_mean,precipitation_sum,windspeed_10m_max",
    "format": "csv"
}

response = requests.get(url, params=params)

with open("../data/istanbul_weather.csv", "w", encoding="utf-8") as f:
    f.write(response.text)

weather = pd.read_csv("../data/istanbul_weather.csv", skiprows=3)
weather.columns = ["date", "temp_mean", "precipitation", "windspeed"]
weather["date"] = pd.to_datetime(weather["date"])
weather = weather.set_index("date")

print("Saved:", len(weather), "days")
weather.head()

Saved: 731 days


,temp_mean,precipitation,windspeed
date,,,
2023-01-01,7.6,0.0,12.2
2023-01-02,6.8,0.0,10.6
2023-01-03,6.9,0.0,17.6
2023-01-04,8.6,0.1,15.9
2023-01-05,9.4,0.0,16.6


## 3. Merge Datasets

In [3]:
df = bist.join(weather, how="inner")
df.to_csv("../data/merged.csv")
print("Merged dataset:", df.shape)
df.head()

Merged dataset: (496, 7)


,open,close,volume,daily_return,temp_mean,precipitation,windspeed
date,,,,,,,
2023-01-02,5568.370125,5661.069824,4834734400,1.664755,6.8,0.0,10.6
2023-01-03,5697.569937,5626.570312,5459192700,-1.246139,6.9,0.0,17.6
2023-01-04,5646.570148,5523.470703,4992602800,-2.180075,8.6,0.1,15.9
2023-01-05,5555.970307,5116.372559,4510533000,-7.912169,9.4,0.0,16.6
2023-01-06,5120.972851,5341.971680,5331359700,4.315563,9.4,4.0,19.5


## 4. Machine Learning

In [4]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, classification_report

X = df[["temp_mean", "precipitation", "windspeed"]]
y = df["daily_return"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# linear regression
model = LinearRegression()
model.fit(X_train, y_train)
preds = model.predict(X_test)
print("R2:", round(model.score(X_test, y_test), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, preds)), 4))

R2: 0.0014
RMSE: 1.6108


In [5]:
# logistic regression - predict up or down
y_dir = (df["daily_return"] > 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y_dir, test_size=0.2, random_state=42)

clf = LogisticRegression()
clf.fit(X_train, y_train)
print(classification_report(y_test, clf.predict(X_test)))

              precision    recall  f1-score   support

           0       0.52      0.96      0.68        52
           1       0.50      0.04      0.08        48

    accuracy                           0.52       100
   macro avg       0.51      0.50      0.38       100
weighted avg       0.51      0.52      0.39       100

